In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif

DATA_PATH = "../data/raw/student_mental_health.csv"
TARGET = "Mental_Health_Risk"

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

df.info()
display(df.sample(5, random_state=42))

In [ ]:
num_cols = df.select_dtypes(include="number").columns.drop(TARGET, errors="ignore")
cat_cols = df.select_dtypes(exclude="number").columns.drop(TARGET, errors="ignore")

print("Negative values per numeric column:")
print((df[num_cols] < 0).sum()[lambda s: s > 0])
print("\nMin / Max:")
print(df[num_cols].agg(["min", "max"]).T)

for col in cat_cols:
    print(f"\n{col}: {df[col].unique()}")

In [ ]:
missing = pd.DataFrame({
    "Count": df.isna().sum(),
    "Percentage": (df.isna().mean() * 100).round(2)
})

print("Missing values:")
display(missing[missing["Count"] > 0])

print("Duplicated rows:", df.duplicated().sum())
print("Missing target:", df[TARGET].isna().sum())

In [ ]:
Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)
IQR = Q3 - Q1

outliers = (
    (df[num_cols] < Q1 - 1.5 * IQR) |
    (df[num_cols] > Q3 + 1.5 * IQR)
)

print("Outliers per column:")
print(outliers.sum().sort_values(ascending=False))

plt.figure(figsize=(18, 10))
sns.boxplot(data=df[num_cols])
plt.title("Outliers Distribution")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
print("Numerical statistics:")
display(df.describe().T)

print("Categorical statistics:")
display(df.describe(include="object").T)

In [ ]:
counts = df[TARGET].value_counts()
print(counts)
print((df[TARGET].value_counts(normalize=True) * 100).round(2))

sns.countplot(x=TARGET, data=df, order=counts.index)
plt.title("Target distribution")
plt.show()

In [ ]:
for col in cat_cols.drop("Student_ID", errors="ignore"):
    sns.countplot(x=col, hue=TARGET, data=df)
    plt.title(f"{col} vs {TARGET}")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
corr = df[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.show()

In [ ]:
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET]

for col in X.columns:
    if X[col].dtype == "object":
        X[col] = X[col].astype("category").cat.codes
    X[col] = X[col].fillna(X[col].median())

scores = pd.Series(mutual_info_classif(X, y, random_state=42), index=X.columns)
scores = scores.sort_values(ascending=False)
print(scores)

scores.plot(kind="barh", figsize=(8, 6)).invert_yaxis()
plt.title("Mutual information with the target")
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

sns.stripplot(
    data=df,
    x="Sleep_Hours",
    y="Mental_Health_Risk",
    order=["Low", "Medium", "High"],
    jitter=0.25,
    alpha=0.4
)

plt.title("Sleep Hours vs Mental Health Risk")
plt.xlabel("Sleep Hours")
plt.ylabel("Mental Health Risk Level")
plt.show()

In [ ]:

plt.figure(figsize=(9, 5))

sns.scatterplot(
    data=df,
    x="Sleep_Hours",
    y="GPA",
    hue="Mental_Health_Risk",
    hue_order=["Low", "Medium", "High"],
    alpha=0.6
)

plt.title("Sleep Hours vs GPA by Mental Health Risk")
plt.xlabel("Sleep Hours")
plt.ylabel("GPA")
plt.show()
